## 1. Set Up

This notebook builds a Conditional Random Field (CRF) part-of-speech tagger for Burmese. The workflow moves from loading the tagged corpus to feature engineering, model training, evaluation, regularization tuning, and error analysis.

The main question is whether a sequence model can use both a word and its surrounding context to assign POS tags reliably.

In [1]:
import time
from collections import Counter, defaultdict
from typing import Sequence
from pathlib import Path
import pickle as plk

import sklearn_crfsuite

ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "dataset"
TRAIN_DIR = DATA_DIR / "mypos-ver.3.0.shuf.nopipe.txt"
TEST_DIR = DATA_DIR / "otest.1k.nopipe.txt"

## 2. Loading the corpus and Spliting the tokens into X(Word) and y(Tag)

In [2]:
def load_corpus(path):
    """Read a myPOS file into sentences of (word, tag) pairs.

    A word may itself contain a slash (dates, fractions), so the tag is always
    split off from the right.
    """
    sentences, invalid_lines = [], []
    with open(path, encoding="utf-8") as corpus:
        for lineno, line in enumerate(corpus, start=1):
            sentence = []
            for token in line.split():
                word, sep, tag = token.rpartition("/")
                if not sep or not word:
                    invalid_lines.append(f"Line {lineno}: Invalid token {token!r}")
                    continue
                sentence.append((word, tag))
            if sentence:
                sentences.append(sentence)
    return sentences, invalid_lines


train, train_invalid_lines = load_corpus(TRAIN_DIR)
test, test_invalid_lines = load_corpus(TEST_DIR)

In [3]:
train[0]

[('၁၉၆၂', 'num'),
 ('ခုနှစ်', 'n'),
 ('ခန့်မှန်း', 'v'),
 ('သန်းခေါင်စာရင်း', 'n'),
 ('အရ', 'ppm'),
 ('လူဦးရေ', 'n'),
 ('၁၁၅၉၃၁', 'num'),
 ('ယောက်', 'part'),
 ('ရှိ', 'v'),
 ('သည်', 'ppm'),
 ('။', 'punc')]

### Dataset stats

In [ ]:
print(f"{'Train corpus':<12}: {len(train):,} sentences, "
      f"{sum(len(s) for s in train):,} tokens, {len(train_invalid_lines)} parsing problems")

print(f"{'Test file':<12}: {len(test):,} sentences, "
      f"{sum(len(s) for s in test):,} tokens, {len(test_invalid_lines)} parsing problems")

print(f"{'Tags':<12}:",
      ", ".join(sorted({t for s in train for _, t in s})))

Train corpus: 43,196 sentences, 564,517 tokens, 0 parse problems
Test file   : 1,000 sentences, 13,468 tokens, 0 parse problems
Tags        : abb, adj, adv, conj, fw, int, n, num, part, ppm, pron, punc, sb, tn, v


The corpus contains **43,196 training sentences and 564,517 training tokens**, plus **1,000 test sentences and 13,468 test tokens**. Both files report **zero parsing problems**, which is a good sign that the token/tag format was read cleanly.

The training data contains 15 POS labels. Since some labels are much less common than others, overall token accuracy alone may hide weaker performance on rare classes. That's why the macro-averaged F1 score and per-label results are useful alongside accuracy.

## 3. Feature Extracion (2-gram method with BOS/EOS markers)

The feature design gives the CRF a local view of each token: the current word, up to two words on either side, neighboring word pairs, and sentence-boundary markers. This is useful because a word's grammatical role often depends on nearby words, not just on the word itself.

One trade-off: these are surface-form features, so words with little or no training evidence may be harder to tag. The model also does not explicitly use morphology or character-level features here.

In [5]:
def baseline_features(sentence, i):
    """Word identities in a +-2 window plus two bigrams (the class template)."""
    words = [w for w, _ in sentence]
    n = len(words)
    feats = []

    for offset in (-2, -1, 0, 1, 2):
        j = i + offset
        if 0 <= j < n:
            feats.append(f"w[{offset}]={words[j]}")

    if i - 1 >= 0:
        feats.append(f"w[-1]|w[0]={words[i - 1]}|{words[i]}")
    if i + 1 < n:
        feats.append(f"w[0]|w[1]={words[i]}|{words[i + 1]}")

    if i == 0:
        feats.append("__BOS__")
    if i == n - 1:
        feats.append("__EOS__")
    return feats


def featurize(sentences, fn):
    X = [[fn(s, i) for i in range(len(s))] for s in sentences]
    y = [[tag for _, tag in s] for s in sentences]
    return X, y

print(baseline_features(train[0], 2))

['w[-2]=၁၉၆၂', 'w[-1]=ခုနှစ်', 'w[0]=ခန့်မှန်း', 'w[1]=သန်းခေါင်စာရင်း', 'w[2]=အရ', 'w[-1]|w[0]=ခုနှစ်|ခန့်မှန်း', 'w[0]|w[1]=ခန့်မှန်း|သန်းခေါင်စာရင်း']


In [6]:
X_train, y_train = featurize(train, baseline_features)
X_test, y_test = featurize(test, baseline_features)

## 4. Training the base model

The model is trained with the L-BFGS optimizer and regularization settings `c1=1.0` and `c2=0.001`. The training time is worth noting because it gives context for later experiments: even a single fit can be relatively expensive, and the parameter search repeats training many times.

In [7]:
def train_crf(X, y, c1, c2):
    crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=c1, c2=c2,
                               max_iterations=200, all_possible_transitions=True)
    crf.fit(X, y)
    return crf

start = time.perf_counter()
crf_base_model = train_crf(X_train, y_train, c1=1.0, c2=1e-3)
print(f"Trained in {time.perf_counter() - start:.1f}s")

Trained in 176.8s


## 5. Evaluation (metrics are in crfsuite format)

In [8]:
def evaluate(
    y_true: Sequence[Sequence[str]],
    y_pred: Sequence[Sequence[str]],
) -> dict:
    """Calculate CRFsuite-style POS-tagging evaluation metrics."""

    if len(y_true) != len(y_pred):
        raise ValueError("Gold and predicted sentence counts differ.")

    if not y_true:
        raise ValueError("Cannot evaluate an empty corpus.")

    gold_counts = Counter()
    pred_counts = Counter()
    match_counts = Counter()

    correct_tokens = 0
    total_tokens = 0
    correct_sentences = 0
    total_sentences = 0

    for sentence_index, (gold, pred) in enumerate(
        zip(y_true, y_pred), start=1
    ):
        if len(gold) != len(pred):
            raise ValueError(
                f"Sentence {sentence_index}: gold has {len(gold)} "
                f"tokens but predictions have {len(pred)}."
            )

        if not gold:
            continue

        total_sentences += 1
        sentence_correct = True

        for true_tag, pred_tag in zip(gold, pred):
            gold_counts[true_tag] += 1
            pred_counts[pred_tag] += 1
            total_tokens += 1

            if true_tag == pred_tag:
                correct_tokens += 1
                match_counts[true_tag] += 1
            else:
                sentence_correct = False

        if sentence_correct:
            correct_sentences += 1

    if total_tokens == 0:
        raise ValueError("Cannot evaluate a corpus with no tokens.")

    # Preserve gold-label order, then append predicted-only labels.
    label_order = list(dict.fromkeys(
        tag for sentence in y_true for tag in sentence
    ))
    seen_labels = set(label_order)

    for sentence in y_pred:
        for tag in sentence:
            if tag not in seen_labels:
                label_order.append(tag)
                seen_labels.add(tag)

    per_label = {}

    for label in label_order:
        match = match_counts[label]
        model = pred_counts[label]
        ref = gold_counts[label]

        precision = match / model if model else 0.0
        recall = match / ref if ref else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )

        per_label[label] = {
            "match": match,
            "model": model,
            "ref": ref,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

    num_labels = len(per_label)

    macro_precision = sum(
        scores["precision"] for scores in per_label.values()
    ) / num_labels

    macro_recall = sum(
        scores["recall"] for scores in per_label.values()
    ) / num_labels

    macro_f1 = sum(
        scores["f1"] for scores in per_label.values()
    ) / num_labels

    return {
        "per_label": per_label,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "item_correct": correct_tokens,
        "item_total": total_tokens,
        "item_accuracy": correct_tokens / total_tokens,
        "instance_correct": correct_sentences,
        "instance_total": total_sentences,
        "instance_accuracy": correct_sentences / total_sentences,
    }


def evaluation_report(metrics: dict) -> None:
    """Print evaluation metrics in CRFsuite-style format."""

    print(
        "Performance by label (#match, #model, #ref) "
        "(precision, recall, F1):"
    )

    for label, scores in metrics["per_label"].items():
        print(
            f"    {label}: "
            f"({scores['match']}, {scores['model']}, "
            f"{scores['ref']}) "
            f"({scores['precision']:.4f}, "
            f"{scores['recall']:.4f}, "
            f"{scores['f1']:.4f})"
        )

    print(
        "Macro-average precision, recall, F1: "
        f"({metrics['macro_precision']:.6f}, "
        f"{metrics['macro_recall']:.6f}, "
        f"{metrics['macro_f1']:.6f})"
    )

    print(
        f"Item accuracy: {metrics['item_correct']} / "
        f"{metrics['item_total']} "
        f"({metrics['item_accuracy']:.4f})"
    )

    print(
        f"Instance accuracy: {metrics['instance_correct']} / "
        f"{metrics['instance_total']} "
        f"({metrics['instance_accuracy']:.4f})"
    )


In [9]:
metrics = evaluate(y_test, crf_base_model.predict(X_test))
evaluation_report(metrics)

Performance by label (#match, #model, #ref) (precision, recall, F1):
    tn: (138, 139, 142) (0.9928, 0.9718, 0.9822)
    n: (2941, 3071, 3000) (0.9577, 0.9803, 0.9689)
    ppm: (2026, 2055, 2060) (0.9859, 0.9835, 0.9847)
    part: (3125, 3192, 3189) (0.9790, 0.9799, 0.9795)
    punc: (1270, 1270, 1270) (1.0000, 1.0000, 1.0000)
    pron: (462, 473, 476) (0.9767, 0.9706, 0.9737)
    v: (1939, 2006, 2010) (0.9666, 0.9647, 0.9656)
    adv: (227, 248, 262) (0.9153, 0.8664, 0.8902)
    conj: (387, 422, 411) (0.9171, 0.9416, 0.9292)
    adj: (319, 353, 366) (0.9037, 0.8716, 0.8873)
    fw: (51, 54, 87) (0.9444, 0.5862, 0.7234)
    abb: (12, 12, 12) (1.0000, 1.0000, 1.0000)
    num: (145, 146, 155) (0.9932, 0.9355, 0.9635)
    sb: (3, 3, 3) (1.0000, 1.0000, 1.0000)
    int: (24, 24, 25) (1.0000, 0.9600, 0.9796)
Macro-average precision, recall, F1: (0.968825, 0.934143, 0.948513)
Item accuracy: 13069 / 13468 (0.9704)
Instance accuracy: 708 / 1000 (0.7080)


The baseline CRF reaches **97.04% token accuracy** and **94.85% macro-F1** on the recorded test results. Its exact-sentence accuracy is **70.8%**, meaning 708 of the 1,000 sentences were tagged perfectly from start to finish.

The strongest baseline F1 scores include punctuation (**1.000**) and several frequent grammatical labels. The clearest weak spot is the `fw` label, with recall of about **58.6%**; `adj` and `adv` are also less reliable than the top-performing labels. This points to uneven performance across tag types, even when overall token accuracy looks high.

## 6. Regularization tuning for CRF

This grid search compares combinations of L1 (`c1`) and L2 (`c2`) regularization. The selected setting is the one with the highest macro-F1 in the recorded results, which gives each POS tag equal weight rather than letting frequent tags dominate the score.

**Evaluation caveat:** the code uses the test corpus to choose the regularization settings and then reports performance on that same corpus. That makes the tuned score a model-selection result, not a fully independent estimate of generalization. For a stronger final evaluation, tune on a separate validation split and use the test set only once at the end.

In [10]:
from itertools import product

c1_values = [0.0, 0.1, 0.5, 1.0, 2.0]
c2_values = [0.0, 1e-5, 1e-4, 1e-3, 1e-2]

results = []
print("Regularization tuning in progress.....")
start = time.perf_counter()
for c1, c2 in product(c1_values, c2_values):
    model = train_crf(X_train, y_train, c1=c1, c2=c2)
    pred = model.predict(X_test)
    metrics = evaluate(y_test, pred)
    results.append({
        'c1': c1,
        'c2': c2,
        'macro_f1': metrics['macro_f1'],
        'item_accuracy': metrics['item_accuracy'],
        'instance_accuracy': metrics['instance_accuracy'],
    })
print(f"Regularization tuning completed in {time.perf_counter() - start:.1f}s\n")

best = max(results, key=lambda r: r["macro_f1"])
print("Best CRF regularization settings:")
for key in ["c1", "c2", "macro_f1", "item_accuracy", "instance_accuracy"]:
    print(f"{key}: {best[key]}")
print()

best_model = train_crf(X_train, y_train, c1=best["c1"], c2=best["c2"])
best_predictions = best_model.predict(X_test)
best_metrics = evaluate(y_test, best_predictions)
evaluation_report(best_metrics)

Regularization tuning in progress.....
Regularization tuning completed in 2308.5s

Best CRF regularization settings:
c1: 0.0
c2: 0.001
macro_f1: 0.9976203003430507
item_accuracy: 0.9967329967329968
instance_accuracy: 0.959

Performance by label (#match, #model, #ref) (precision, recall, F1):
    tn: (142, 142, 142) (1.0000, 1.0000, 1.0000)
    n: (2995, 3000, 3000) (0.9983, 0.9983, 0.9983)
    ppm: (2054, 2056, 2060) (0.9990, 0.9971, 0.9981)
    part: (3175, 3188, 3189) (0.9959, 0.9956, 0.9958)
    punc: (1270, 1270, 1270) (1.0000, 1.0000, 1.0000)
    pron: (471, 472, 476) (0.9979, 0.9895, 0.9937)
    v: (2000, 2016, 2010) (0.9921, 0.9950, 0.9935)
    adv: (261, 261, 262) (1.0000, 0.9962, 0.9981)
    conj: (409, 410, 411) (0.9976, 0.9951, 0.9963)
    adj: (365, 371, 366) (0.9838, 0.9973, 0.9905)
    fw: (87, 87, 87) (1.0000, 1.0000, 1.0000)
    abb: (12, 12, 12) (1.0000, 1.0000, 1.0000)
    num: (155, 155, 155) (1.0000, 1.0000, 1.0000)
    sb: (3, 3, 3) (1.0000, 1.0000, 1.0000)
    int

The recorded search selects **`c1=0.0` and `c2=0.001`**, with **99.76% macro-F1**, **99.67% token accuracy**, and **95.9% exact-sentence accuracy**. The error analysis below reports 44 remaining token mistakes for this selected model, compared with 399 for the baseline.

One detail worth calling out: the best setting has no L1 penalty (`c1=0.0`) and a small L2 penalty. Within this search grid, that combination performed best. Because the test set was used to select it, though, the result should be confirmed on a validation split before treating the score as the model's expected performance on unseen data.

## 7. Error analysis

### CRF model before regularization tuning

In [11]:
y_test_pred = crf_base_model.predict(X_test)

errors = [
    {"word": word, "gold": gold, "pred": pred}
    for sentence, truth, prediction in zip(test, y_test, y_test_pred)
    for (word, _), gold, pred in zip(sentence, truth, prediction)
    if gold != pred
]

seen = {}
for sentence in train:
    for word, tag in sentence:
        seen.setdefault(word, Counter())[tag] += 1

unseen = sum(1 for e in errors if e["word"] not in seen)

print(f"Total errors: {len(errors):,}")
if errors:
    print(f"Errors on words never seen in training: {unseen} ({unseen / len(errors):.1%})")

print("\nMost common confusions (gold -> predicted):")
for (gold, pred), n in Counter(
    (e["gold"], e["pred"]) for e in errors
).most_common(10):
    print(f"  {gold:>5} -> {pred:<5} {n:>4}")

Total errors: 399
Errors on words never seen in training: 0 (0.0%)

Most common confusions (gold -> predicted):
     fw -> n       35
      v -> n       29
      v -> part    27
   part -> v       22
      n -> v       21
    adv -> n       20
    ppm -> conj    16
   part -> conj    15
      n -> part    14
    adj -> v       14


Before tuning, the error analysis finds **399 token-level mistakes**, and none of those errors are on words absent from the training vocabulary. In this test sample, the mistakes therefore come from words the model has seen before, suggesting that ambiguity or contextual decisions are a larger issue than completely unseen word forms.

The most frequent confusions include `fw → n`, `v → n`, and `v → part`. These patterns suggest that some labels can be difficult to distinguish from local word context alone. Looking at examples from these confusion pairs would be a useful next step.

### Best CRF model selected after regularization tuning

In [12]:
y_test_pred = best_model.predict(X_test)

errors = [
    {"word": word, "gold": gold, "pred": pred}
    for sentence, truth, prediction in zip(test, y_test, y_test_pred)
    for (word, _), gold, pred in zip(sentence, truth, prediction)
    if gold != pred
]

seen = {}
for sentence in train:
    for word, tag in sentence:
        seen.setdefault(word, Counter())[tag] += 1

unseen = sum(1 for e in errors if e["word"] not in seen)

print(f"Total errors: {len(errors):,}")
if errors:
    print(f"Errors on words never seen in training: {unseen} ({unseen / len(errors):.1%})")

print("\nMost common confusions (gold -> predicted):")
for (gold, pred), n in Counter(
    (e["gold"], e["pred"]) for e in errors
).most_common(10):
    print(f"  {gold:>5} -> {pred:<5} {n:>4}")

Total errors: 44
Errors on words never seen in training: 0 (0.0%)

Most common confusions (gold -> predicted):
   part -> v       12
      v -> part     7
    ppm -> part     3
   pron -> adj      3
    ppm -> n        2
      n -> part     2
   pron -> n        2
      v -> adj      2
      n -> v        2
      n -> pron     1


After selecting the tuned model, the recorded error count falls to **44**, compared with 399 for the baseline. The most frequent remaining confusion is `part → v` (12 cases), followed by `v → part` (7 cases). These two labels account for a noticeable share of the remaining mistakes.

The improvement is substantial in the current results, but keep the test-set tuning caveat in mind: because the same test corpus was used to select the settings, this comparison may overstate performance on new data. A separate validation set would help confirm how much of the gain carries over.

## Overall takeaway

The CRF baseline already performs strongly, with **97.04% token accuracy** and **94.85% macro-F1** in the recorded evaluation. The tuned run reports a much higher macro-F1 of **99.76%** and token accuracy of **99.67%**, with 44 errors remaining in the error analysis.

The main practical takeaway is that contextual sequence features appear useful for Burmese POS tagging, while confusions between `v` and `part` remain worth investigating. Because the test set was used during tuning, treat the tuned figures as encouraging but not yet as a final unbiased benchmark; validate the settings on separate data before making a firm performance claim.

## 8. Saving the best model

In [13]:
MODEL_DIR = ROOT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "crf_pos.pkl"

with MODEL_PATH.open("wb") as model_file:
    plk.dump(best_model, model_file)

print(f"Best CRF model saved to: {MODEL_PATH}")

Best CRF model saved to: c:\Users\onest\OneDrive\Desktop\Projects\BiLSTM_CRF_Burmese_POS_Tagger\models\crf_pos.pkl


In [15]:
def predict_sentence(text, model=best_model):
    words = text.split()
    if not words:
        return []

    sentence = [(word, "") for word in words]
    features = [baseline_features(sentence, i) for i in range(len(words))]
    tags = model.predict([features])[0]
    return list(zip(words, tags))

text = input("Enter a whitespace-tokenized sentence: ").strip()
if text:
    tagged_sentence = predict_sentence(text)
    print(" ".join(f"{word}/{tag}" for word, tag in tagged_sentence))
else:
    print("No sentence entered.")


အပြင်/n သွား/v မ/part လား/part


The interactive example is a quick sanity check that the saved model can tag a whitespace-tokenized Burmese sentence. The function assumes the input is already tokenized; it does not split punctuation or perform sentence segmentation. For real-world use, that preprocessing step needs to match the format expected by the model.